In [3]:
import sqlite3
import pandas as pd

In [4]:
import os

BASE_DIR = os.path.expanduser('~/Desktop/Customer Lifecycle Analysis Project')

print(f'Project root : {BASE_DIR}')
print(f'Data folder  : {os.path.join(BASE_DIR, "data")}')
print(f'Folders exist: data={os.path.exists(os.path.join(BASE_DIR, "data"))}, outputs={os.path.exists(os.path.join(BASE_DIR, "outputs"))}')

Project root : /Users/amrit/Desktop/Customer Lifecycle Analysis Project
Data folder  : /Users/amrit/Desktop/Customer Lifecycle Analysis Project/data
Folders exist: data=True, outputs=True


In [5]:
# Connect to SQLite database (creates the file if it doesn't exist)
DB_PATH  = os.path.join(BASE_DIR, 'data', 'customer_lifecycle.db')
conn = sqlite3.connect(DB_PATH)

# Load CSVs
rfm = pd.read_csv(os.path.join(BASE_DIR, 'data', 'rfm_with_churn.csv'), dtype={'Customer ID': str})
df  = pd.read_csv(os.path.join(BASE_DIR, 'data', 'online_retail_clean.csv'), dtype={'Customer ID': str}, parse_dates=['InvoiceDate'])

# Write both tables into the database
rfm.to_sql('rfm_segments', conn, if_exists='replace', index=False)
df.to_sql('transactions',  conn, if_exists='replace', index=False)

print(f'Database created : {DB_PATH}')
print(f'Tables loaded    : rfm_segments ({len(rfm):,} rows), transactions ({len(df):,} rows)')

Database created : /Users/amrit/Desktop/Customer Lifecycle Analysis Project/data/customer_lifecycle.db
Tables loaded    : rfm_segments (5,878 rows), transactions (779,425 rows)


## SQL Queries

All queries run against a local SQLite database containing two tables:
- `rfm_segments` — 5,878 customers with RFM scores, segments and churn predictions
- `transactions` — 779,425 clean transaction rows

A helper function is used to run each query and return results as a pandas DataFrame.

In [6]:
def run_query(sql, description=''):
    result = pd.read_sql_query(sql, conn)
    if description:
        print(f'── {description} ──')
    print(result.to_string(index=False))
    print(f'\n{len(result)} rows returned\n')
    return result

In [7]:
run_query("""
    SELECT 
        Segment,
        COUNT(*) as customer_count,
        ROUND(SUM(Monetary), 2) as total_revenue,
        ROUND(AVG(Monetary), 2) as avg_revenue
    FROM rfm_segments
    GROUP BY Segment
    ORDER BY total_revenue DESC
""", 'Query 1 — Segment Revenue Summary')

── Query 1 — Segment Revenue Summary ──
        Segment  customer_count  total_revenue  avg_revenue
      Champions            1482    12024330.14      8113.58
Loyal Customers            1221     2510046.34      2055.73
 Need Attention             551     1129337.08      2049.61
           Lost            1523      654426.70       429.70
Potential Loyal             828      596616.76       720.55
        At Risk              89      269452.77      3027.56
  New Customers             184      190594.46      1035.84

7 rows returned



,Segment,customer_count,total_revenue,avg_revenue
0,Champions,1482,12024330.14,8113.58
1,Loyal Customers,1221,2510046.34,2055.73
2,Need Attention,551,1129337.08,2049.61
3,Lost,1523,654426.70,429.70
4,Potential Loyal,828,596616.76,720.55
5,At Risk,89,269452.77,3027.56
6,New Customers,184,190594.46,1035.84


In [8]:
run_query("""
SELECT 
    Segment,
    ROUND(SUM(Monetary), 2) as revenue,
    ROUND(SUM(Monetary) * 100.0 / SUM(SUM(Monetary)) OVER (), 1) as revenue_pct
FROM rfm_segments
GROUP BY Segment
ORDER BY revenue DESC

""", 'Query 2 - Revenue Concentration by Segment')

── Query 2 - Revenue Concentration by Segment ──
        Segment     revenue  revenue_pct
      Champions 12024330.14         69.2
Loyal Customers  2510046.34         14.4
 Need Attention  1129337.08          6.5
           Lost   654426.70          3.8
Potential Loyal   596616.76          3.4
        At Risk   269452.77          1.6
  New Customers   190594.46          1.1

7 rows returned



,Segment,revenue,revenue_pct
0,Champions,12024330.14,69.2
1,Loyal Customers,2510046.34,14.4
2,Need Attention,1129337.08,6.5
3,Lost,654426.70,3.8
4,Potential Loyal,596616.76,3.4
5,At Risk,269452.77,1.6
6,New Customers,190594.46,1.1


In [9]:
run_query("""
SELECT 
    "Customer ID",
    Recency,
    Frequency,
    ROUND(Monetary, 2) AS Monetary,
    Segment
FROM rfm_segments
ORDER BY Monetary DESC
LIMIT 10
""", 'Query 3 — Top 10 Customers by Lifetime Value')

── Query 3 — Top 10 Customers by Lifetime Value ──
Customer ID  Recency  Frequency  Monetary         Segment
      18102        1        145 580987.04       Champions
      14646        2        151 528602.52       Champions
      14156       10        156 313437.62       Champions
      14911        1        398 291420.81       Champions
      17450        8         51 244784.25       Champions
      13694        4        143 195640.69       Champions
      17511        3         60 172132.87       Champions
      16446        1          2 168472.50 Potential Loyal
      16684        4         55 147142.77       Champions
      12415       24         28 144458.37       Champions

10 rows returned



,Customer ID,Recency,Frequency,Monetary,Segment
0,18102,1,145,580987.04,Champions
1,14646,2,151,528602.52,Champions
2,14156,10,156,313437.62,Champions
3,14911,1,398,291420.81,Champions
4,17450,8,51,244784.25,Champions
5,13694,4,143,195640.69,Champions
6,17511,3,60,172132.87,Champions
7,16446,1,2,168472.50,Potential Loyal
8,16684,4,55,147142.77,Champions
9,12415,24,28,144458.37,Champions


In [ ]:
run_query("""
SELECT 
    t.InvoiceMonth,
    COUNT(DISTINCT t.Invoice) AS orders,
    ROUND(SUM(t.TotalRevenue), 2) AS revenue
FROM transactions t
JOIN rfm_segments r ON t."Customer ID" = r."Customer ID"
GROUP BY t.InvoiceMonth
ORDER BY t.InvoiceMonth
""", 'Query 4 — Monthly Revenue Trend')

In [ ]:
run_query("""
SELECT 
    r.Segment,
    ROUND(AVG(t.Price), 2) AS avg_unit_price,
    ROUND(AVG(t.Quantity), 2) AS avg_quantity,
    COUNT(DISTINCT t.StockCode) AS unique_products_bought
FROM transactions t
JOIN rfm_segments r ON t."Customer ID" = r."Customer ID"
WHERE r.Segment = 'Champions'
GROUP BY r.Segment
""", 'Query 5 — Champion Purchasing Patterns')

In [ ]:
run_query("""
SELECT
    "Customer ID",
    Recency,
    Frequency,
    ROUND(Monetary, 2) AS Monetary
FROM rfm_segments
WHERE Segment = 'At Risk' AND Monetary > 1000
ORDER BY Monetary DESC
""", 'Query 6 — High Value At Risk Customers')

In [10]:
run_query("""
SELECT 
    t.InvoiceMonth,
    COUNT(DISTINCT t.Invoice) AS orders,
    ROUND(SUM(t.TotalRevenue), 2) AS revenue
FROM transactions t
JOIN rfm_segments r ON t."Customer ID" = r."Customer ID"
GROUP BY t.InvoiceMonth
ORDER BY t.InvoiceMonth
""", 'Query 4 — Monthly Revenue Trend')

── Query 4 — Monthly Revenue Trend ──
InvoiceMonth  orders    revenue
     2009-12    1512  683504.01
     2010-01    1011  555802.67
     2010-02    1104  504558.96
     2010-03    1524  696978.47
     2010-04    1329  591982.00
     2010-05    1377  597833.38
     2010-06    1497  636371.13
     2010-07    1381  589736.17
     2010-08    1293  602224.60
     2010-09    1689  829013.95
     2010-10    2133 1033112.01
     2010-11    2587 1166460.02
     2010-12    1400  570422.73
     2011-01     987  568101.31
     2011-02     997  446084.92
     2011-03    1321  594081.76
     2011-04    1149  468374.33
     2011-05    1555  677355.15
     2011-06    1393  660046.05
     2011-07    1331  598962.90
     2011-08    1280  644051.04
     2011-09    1755  950690.20
     2011-10    1929 1035642.45
     2011-11    2657 1156205.61
     2011-12     778  517208.44

25 rows returned



,InvoiceMonth,orders,revenue
0,2009-12,1512,683504.01
1,2010-01,1011,555802.67
2,2010-02,1104,504558.96
3,2010-03,1524,696978.47
4,2010-04,1329,591982.00
5,2010-05,1377,597833.38
6,2010-06,1497,636371.13
7,2010-07,1381,589736.17
8,2010-08,1293,602224.60
9,2010-09,1689,829013.95


In [11]:
run_query("""
SELECT 
    r.Segment,
    ROUND(AVG(t.Price), 2) AS avg_unit_price,
    ROUND(AVG(t.Quantity), 2) AS avg_quantity,
    COUNT(DISTINCT t.StockCode) AS unique_products_bought
FROM transactions t
JOIN rfm_segments r ON t."Customer ID" = r."Customer ID"
WHERE r.Segment = 'Champions'
GROUP BY r.Segment
""", 'Query 5 — Champion Purchasing Patterns')

── Query 5 — Champion Purchasing Patterns ──
  Segment  avg_unit_price  avg_quantity  unique_products_bought
Champions            3.11         14.06                    4539

1 rows returned



,Segment,avg_unit_price,avg_quantity,unique_products_bought
0,Champions,3.11,14.06,4539


In [12]:
run_query("""
SELECT
    "Customer ID",
    Recency,
    Frequency,
    ROUND(Monetary, 2) AS Monetary
FROM rfm_segments
WHERE Segment = 'At Risk' AND Monetary > 1000
ORDER BY Monetary DESC
""", 'Query 6 — High Value At Risk Customers')

── Query 6 — High Value At Risk Customers ──
Customer ID  Recency  Frequency  Monetary
      13902      632          5  34095.26
      12482      576         29  23691.40
      14063      438          9  22710.20
      17448      497         46  14523.67
      14160      610          7   8421.47
      15413      692          5   6798.72
      15369      492         11   6251.26
      12835      428         41   5996.83
      14249      411         12   5400.46
      14685      576          8   4536.64
      14648      416         15   4466.65
      15633      509         13   4335.96
      14025      465          8   4057.90
      13446      430         11   3881.89
      16636      438          4   3802.04
      16336      421          4   3517.69
      15538      538          6   3358.40
      13214      418          7   3264.23
      17145      513          4   3120.91
      14590      425         20   2993.37
      17032      548          8   2984.32
      16736      506          5

,Customer ID,Recency,Frequency,Monetary
0,13902,632,5,34095.26
1,12482,576,29,23691.40
2,14063,438,9,22710.20
3,17448,497,46,14523.67
4,14160,610,7,8421.47
...,...,...,...,...
64,16702,415,4,1110.83
65,17137,416,6,1099.81
66,13613,453,5,1039.85
67,15003,473,6,1027.03


In [14]:
run_query("""SELECT 
    t.Country,
    r.Segment,
    COUNT(DISTINCT r."Customer ID") as customers,
    ROUND(SUM(t.TotalRevenue), 2) as revenue
FROM transactions t
JOIN rfm_segments r ON t."Customer ID" = r."Customer ID"
GROUP BY t.Country, r.Segment
HAVING COUNT(DISTINCT r."Customer ID") > 10
ORDER BY revenue DESC
""", 'Query 7 — Country & Segment Breakdown')

── Query 7 — Country & Segment Breakdown ──
       Country         Segment  customers    revenue
United Kingdom       Champions       1353 9727585.74
United Kingdom Loyal Customers       1116 2180772.62
United Kingdom  Need Attention        514 1048946.10
United Kingdom            Lost       1380  529152.03
United Kingdom Potential Loyal        725  513830.47
       Germany       Champions         33  319940.88
        France       Champions         31  272902.98
United Kingdom         At Risk         85  208685.85
United Kingdom   New Customers        177  180262.11
       Germany Loyal Customers         29   54589.94
        France Loyal Customers         18   52513.19
       Germany            Lost         22   26964.07
       Germany Potential Loyal         18   12384.29
        France            Lost         20   10359.93
        France Potential Loyal         22   10284.32
         Spain Potential Loyal         12    7698.98
         Spain            Lost         12    5534.49

1

,Country,Segment,customers,revenue
0,United Kingdom,Champions,1353,9727585.74
1,United Kingdom,Loyal Customers,1116,2180772.62
2,United Kingdom,Need Attention,514,1048946.10
3,United Kingdom,Lost,1380,529152.03
4,United Kingdom,Potential Loyal,725,513830.47
5,Germany,Champions,33,319940.88
6,France,Champions,31,272902.98
7,United Kingdom,At Risk,85,208685.85
8,United Kingdom,New Customers,177,180262.11
9,Germany,Loyal Customers,29,54589.94


In [15]:
run_query("""SELECT 
    Segment,
    COUNT(*) as total_customers,
    SUM(Churn_Predicted) as predicted_churners,
    ROUND(AVG(Churn_Probability) * 100, 1) as avg_churn_probability_pct
FROM rfm_segments
GROUP BY Segment
ORDER BY avg_churn_probability_pct DESC
""", 'Query 8 — Predicted Churners by Segment')

── Query 8 — Predicted Churners by Segment ──
        Segment  total_customers  predicted_churners  avg_churn_probability_pct
           Lost             1523                1523                       73.5
Potential Loyal              828                 828                       71.3
  New Customers              184                 145                       55.9
 Need Attention              551                 235                       46.0
Loyal Customers             1221                 529                       44.5
        At Risk               89                   4                       35.2
      Champions             1482                  11                       23.0

7 rows returned



,Segment,total_customers,predicted_churners,avg_churn_probability_pct
0,Lost,1523,1523,73.5
1,Potential Loyal,828,828,71.3
2,New Customers,184,145,55.9
3,Need Attention,551,235,46.0
4,Loyal Customers,1221,529,44.5
5,At Risk,89,4,35.2
6,Champions,1482,11,23.0


In [16]:
conn.close()
print('Database connection closed ✓')

Database connection closed ✓


## Summary

Eight SQL queries run against a local SQLite database built from the cleaned 
transaction and RFM segment data produced in Notebooks 01 and 02.

Queries range from simple aggregations to multi-table JOINs, window functions 
for revenue concentration, and cross-referencing churn predictions from 
Notebook 03 against segment data.

Key finding: UK Champions (1,353 customers) generate £9.7M — over half of 
total revenue from a single country-segment combination. Lost customers 
represent 25.9% of the customer base but only 3.8% of revenue, confirming 
the business case for differentiated CRM strategy built across this project.